# EDA Comparativo y Verificación de Calidad del Preprocesamiento de EmoEvent

Este notebook verifica la calidad del pipeline de preprocesamiento aplicado al dataset EmoEvent:
- Normalización de texto (eliminación de HASHTAG/USER/URL, conversión de emojis)
- Traducción ES --> EN de clases minoritarias (fear, disgust, surprise)
- Generación de datos sintéticos para balanceo de clases
- Splits train/val/test sin data leakage

## 0. Imports y configuración

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import re
from pathlib import Path

def save_custom_plot(filename, dpi=300):
    """
    Guarda la figura activa de matplotlib en el directorio de resultados.
    Asegura alta resolución y recorta los márgenes blancos innecesarios.
    """
    output_dir = Path('../docs/figures')
    
    # Aseguramos que la extensión .png esté presente
    if not filename.endswith('.png'):
        filename += '.png'
        
    plt.savefig(output_dir / filename, bbox_inches='tight', dpi=dpi)
    print(f"Gráfica guardada: {filename}")

# Estilo global
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.15)
plt.rcParams.update({'figure.dpi': 130, 'figure.facecolor': 'white'})

DATA_ROOT = Path('../data')
RAW_ES   = DATA_ROOT / 'raw/emoevent/emoevent_es.csv'
RAW_EN   = DATA_ROOT / 'raw/emoevent/emoevent_en.csv'
PREP_CSV = DATA_ROOT / 'processed/emoevent_preprocessed.csv'
SYNT_CSV = DATA_ROOT / 'processed/emoevent_preprocessed_synthetic.csv'
NS_DIR   = DATA_ROOT / 'processed/no_synthetic'
WS_DIR   = DATA_ROOT / 'processed/with_synthetic'

# Paleta de emociones
EMOTION_ORDER = ['anger', 'disgust', 'fear', 'joy', 'sadness', 'surprise', 'others']
PALETTE = sns.color_palette('tab10', n_colors=len(EMOTION_ORDER))
EMO_PALETTE = dict(zip(EMOTION_ORDER, PALETTE))

## 1. Carga de los datasets

In [ ]:
# Carga de datos
raw_es   = pd.read_csv(RAW_ES, sep='\t')
raw_en   = pd.read_csv(RAW_EN, sep='\t')
prep     = pd.read_csv(PREP_CSV)
synt_full = pd.read_csv(SYNT_CSV)

ns_train = pd.read_csv(NS_DIR / 'emoevent_train.csv')
ns_val   = pd.read_csv(NS_DIR / 'emoevent_val.csv')
ns_test  = pd.read_csv(NS_DIR / 'emoevent_test.csv')

ws_train = pd.read_csv(WS_DIR / 'emoevent_train.csv')
ws_val   = pd.read_csv(WS_DIR / 'emoevent_val.csv')
ws_test  = pd.read_csv(WS_DIR / 'emoevent_test.csv')

print(f'raw_es: {raw_es.shape}')
print(f'raw_en: {raw_en.shape}')
print(f'prep: {prep.shape}')
print(f'synt_full: {synt_full.shape}')
print(f'ns  train/val/test: {ns_train.shape} / {ns_val.shape} / {ns_test.shape}')
print(f'ws  train/val/test: {ws_train.shape} / {ws_val.shape} / {ws_test.shape}')

## 2. Comparativa de Distribución de Clases

In [ ]:
# Función para generar métricas de desequilibrio
def imbalance_table(df: pd.DataFrame, label: str) -> pd.DataFrame:
    counts = df['emotion'].value_counts().reindex(EMOTION_ORDER, fill_value=0)
    total = counts.sum()
    pct = (counts / total * 100).round(2)
    
    # Evitar división por cero sustituyendo 0 por NaN para el cálculo del ratio
    min_count = counts.replace(0, np.nan)
    ratio = (counts.max() / min_count).round(2)
    
    return pd.DataFrame({'emoción': EMOTION_ORDER,
                         f'n_{label}': counts.values,
                         f'%_{label}': pct.values,
                         f'ratio_{label}': ratio.values}).set_index('emoción')

# Generación de la tabla comparativa
tbl_ns = imbalance_table(ns_train, 'sin_sint')
tbl_ws = imbalance_table(ws_train, 'con_sint')
tbl_comp = pd.concat([tbl_ns, tbl_ws], axis=1)

print('Tabla Comparativa en Train (Sin vs Con Sintéticos)')
display(tbl_comp.style
        .format({
            '%_sin_sint': '{:.2f}%', '%_con_sint': '{:.2f}%', 
            'ratio_sin_sint': '{:.2f}x', 'ratio_con_sint': '{:.2f}x'
        }, na_rep='-')
        .background_gradient(cmap='Blues', subset=[c for c in tbl_comp.columns if c.startswith('n_')])
        .set_caption('Distribución de clases en el conjunto de entrenamiento (Train)'))

In [ ]:
#  Barplot comparativo lado a lado
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)

for ax, df, title in zip(axes,
                         [ns_train, ws_train],
                         ['Train — SIN sintéticos', 'Train — CON sintéticos']):
    
    counts = df['emotion'].value_counts().reindex(EMOTION_ORDER, fill_value=0)
    total = counts.sum()
    colors = [EMO_PALETTE[e] for e in EMOTION_ORDER]
    
    bars = ax.bar(EMOTION_ORDER, counts.values, color=colors, edgecolor='white', linewidth=0.8)
    
    # Colocación automática de etiquetas en las barras
    labels = [f'{n}\n({n/total*100:.1f}%)' if n > 0 else '0' for n in counts.values]
    ax.bar_label(bars, labels=labels, padding=3, fontsize=8.5)
    
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Emoción')
    ax.set_ylabel('Número de muestras')
    ax.set_ylim(0, counts.max() * 1.25)
    ax.tick_params(axis='x', rotation=20)
    sns.despine(ax=ax)

plt.suptitle('Distribución de clases en Train — Comparativa', fontsize=13, y=1.02)
plt.tight_layout()
save_custom_plot('train_class_comparison')
plt.show()

In [ ]:
# Extracción de métricas puras. Se usa el mínimo de las clases presentes (mayor a 0)
min_ns = ns_train['emotion'].value_counts().min()
min_ws = ws_train['emotion'].value_counts().min()
max_ns = ns_train['emotion'].value_counts().max()
max_ws = ws_train['emotion'].value_counts().max()

ratio_ns = max_ns / min_ns
ratio_ws = max_ws / min_ws
reduction_pct = (1 - ratio_ws/ratio_ns) * 100

print('Métricas Globales de Desequilibrio')
print(f'Ratio Global (Max/Min) SIN sintéticos: {ratio_ns:.2f}x')
print(f'Ratio Global (Max/Min) CON sintéticos: {ratio_ws:.2f}x')
print(f'Reducción matemática del desequilibrio: {reduction_pct:.1f}%')

La inyección de datos sintéticos logró reducir el desequilibrio global en un 18.8%, mitigando la brecha de la clase minoritaria crítica (fear) de 7.34x a 5.96x. Verificaremos si las frases sintéticas son válidas para usarlas en el conjunto de entrenamiento. 

## 3. Verificación de Normalización

In [ ]:
# 1. Muestreo Cualitativo: Original vs Preprocesado
prep_orig = prep[prep['source'] == 'original'].copy()
raw_es_indexed = raw_es.set_index('id')

sample_ids = prep_orig['id'].sample(10, random_state=42).values

rows = []
for sid in sample_ids:
    raw_text = raw_es_indexed.loc[sid, 'tweet'] if sid in raw_es_indexed.index else '—'
    clean_text = prep_orig.loc[prep_orig['id'] == sid, 'text'].values[0]
    rows.append({'id': sid, 'texto_original': raw_text, 'texto_limpio': clean_text})

df_compare = pd.DataFrame(rows)

print('Muestreo Cualitativo: Ruido de Red Social (Menciones, Enlaces)')
# Usamos Styler para que el texto largo no se trunque y sea fácil de leer
display(df_compare.style.set_properties(**{'text-align': 'left', 'white-space': 'pre-wrap'}))

In [ ]:
# Verificación Cuantitativa: Purga de Placeholders de Anonimización
tokens_to_check = ['HASHTAG', 'USER', 'URL']

print('Verificación Estricta: Ocurrencias de Placeholders')
for token in tokens_to_check:
    n = prep['text'].str.contains(token, regex=False).sum()
    status = 'OK (0)' if n == 0 else f'ENCONTRADO ({n} filas)'
    print(f'  {token:10s}: {status}')

In [ ]:
# Verificación Estructural: Conversión a Texto Descriptivo (Demojize)
EMOJI_PATTERN = re.compile(
    '[\U0001F300-\U0001F9FF\U00002600-\U000027BF\U0001FA00-\U0001FA9F]'
)

raw_es_with_emoji = raw_es[raw_es['tweet'].apply(lambda t: bool(EMOJI_PATTERN.search(str(t))))].copy()
ids_with_emoji = set(raw_es_with_emoji['id'].values)

prep_orig_emoji = prep_orig[prep_orig['id'].isin(ids_with_emoji)]

print('Verificación de Traducción de Emojis=')
print(f'Tweets originales detectados con emoji : {len(raw_es_with_emoji)}')
print(f'Muestras preservadas tras undersampling : {len(prep_orig_emoji)}')
print()

# Muestreo cualitativo de la transformación de emojis
sample_emoji_ids = prep_orig_emoji['id'].sample(min(5, len(prep_orig_emoji)), random_state=112).values
examples = []
for sid in sample_emoji_ids:
    raw_t = raw_es_indexed.loc[sid, 'tweet'] if sid in raw_es_indexed.index else '—'
    clean_t = prep_orig_emoji.loc[prep_orig_emoji['id'] == sid, 'text'].values[0]
    examples.append({'id': sid, 'original_con_emoji': raw_t, 'convertido': clean_t})

display(pd.DataFrame(examples).style.set_properties(**{'text-align': 'left'}))

El preprocesamiento eliminó con éxito el 100% de los metadatos residuales (HASHTAG, USER, URL) y transcribió correctamente los emojis a su equivalente textual descriptivo, homogeneizando el corpus sin destruir la señal emocional subyacente.

## 4. Verificación de Traducción (MarianMT)

In [ ]:
translated = prep[prep['source'] == 'translated_en'].copy()
raw_en_indexed = raw_en.set_index('id')

print('Muestreo Cruzado: Original (EN) vs Traducción (ES)')

for emotion in ['fear', 'disgust', 'surprise']:
    subset = translated[translated['emotion'] == emotion]
    n_sample = min(15, len(subset))
    
    if n_sample == 0:
        continue
        
    sample = subset.sample(n_sample, random_state=42)
    
    rows = []
    for _, row in sample.iterrows():
        sid = row['id']
        en_text = raw_en_indexed.loc[sid, 'tweet'] if sid in raw_en_indexed.index else '—'
        rows.append({
            'id': int(sid), 
            'inglés_original': en_text, 
            'español_traducido': row['text']
        })
    
    df_pairs = pd.DataFrame(rows)
    print(f'\n[ Emoción: {emotion.upper()}, Muestras: {n_sample} ]')
    
    # Aplicamos estilo para facilitar la lectura de textos largos
    display(df_pairs.style.hide(axis="index") .set_properties(**{'text-align': 'left', 'white-space': 'pre-wrap', 'vertical-align': 'top'}))

La inspección visual confirma que el modelo MarianMT logró transferir de manera efectiva la intensidad emocional, el tono exclamativo y la semántica subyacente de las clases minoritarias del inglés al español.

## 5. Verificación de Datos Sintéticos

In [ ]:
# Extracción de muestras sintéticas en el conjunto de entrenamiento
synthetic = ws_train[ws_train['source'] == 'synthetic'].copy()
print('Volumen de Datos Sintéticos en Train')
print(synthetic['emotion'].value_counts().to_string())

In [ ]:
# Muestreo Cualitativo (Fear y Disgust)
print('\nMuestreo Cualitativo: Textos Sintéticos (Mistral 7B)')
for emotion in ['fear', 'disgust']:
    sub = synthetic[synthetic['emotion'] == emotion]
    n = min(15, len(sub))
    if n == 0:
        continue
        
    print(f'\n[ Sintéticos: {emotion.upper()}, Muestras: {n} ]')
    display(sub[['id', 'text']].sample(n, random_state=112).reset_index(drop=True).style.hide(axis="index").set_properties(**{'text-align': 'left', 'white-space': 'pre-wrap'}))

In [ ]:
# Análisis Dimensional (Longitud de textos)
def word_count(text: str) -> int:
    return len(str(text).split())

ws_train['n_words'] = ws_train['text'].apply(word_count)

# Mapeo descriptivo para las leyendas
ws_train['tipo'] = ws_train['source'].map({
    'original': 'Original',
    'translated_en': 'Traducido (EN --> ES)',
    'synthetic': 'Sintético (Ollama)'
})

# Tabla pivote de promedios
mean_table = ws_train.groupby(['emotion', 'tipo'])['n_words'].mean().round(1).unstack()
print('Longitud Media (Palabras) por Emoción y Fuente')
display(mean_table.style.background_gradient(cmap='YlOrRd', axis=None).format(na_rep='-'))

In [ ]:
# Boxplot de distribuciones comparadas
order_tipo = ['Original', 'Traducido (EN --> ES)', 'Sintético (Ollama)']
palette_tipo = {'Original': '#4878CF', 'Traducido (EN --> ES)': '#6ACC65', 'Sintético (Ollama)': '#D65F5F'}

fig, ax = plt.subplots(figsize=(12, 5))
sns.boxplot(
    data=ws_train[ws_train['tipo'].notna()],
    x='emotion', y='n_words', hue='tipo',
    order=EMOTION_ORDER, hue_order=order_tipo,
    palette=palette_tipo,
    flierprops=dict(marker='o', markersize=3, alpha=0.4),
    ax=ax
)

ax.set_title('Distribución de Longitud (Palabras) por Emoción y Fuente', fontweight='bold', pad=12)
ax.set_xlabel('Emoción', labelpad=8)
ax.set_ylabel('Número de palabras', labelpad=8)
ax.legend(title='Fuente de Datos', bbox_to_anchor=(1.01, 1), loc='upper left')
sns.despine(ax=ax)
plt.tight_layout()

save_custom_plot('length_boxplot_by_source')
plt.show()

**Esta inspección nos da el argumento para descartar los datos sintéticos.** Los textos generados por Mistral caen en un patrón de evidente (repitiendo fórmulas exactas como "¿Qué hago si..." o "El ciberacoso es una maldición..."). Además, el boxplot expone que la varianza de la longitud en los datos sintéticos es casi nula (la caja roja está aplastada) frente a la dispersión asimétrica y natural de los datos humanos y traducidos. Entrenar el clasificador con este corpus sintético provocaría un fuerte sobreajuste (overfitting), haciendo que el modelo aprenda a detectar los patrones, en lugar de entender el lenguaje ruidoso y real del ciberacoso.

## 6. Verificación de Fuga de Datos y Estratificación al corpus que usaremos (Datos Reales + Traducidos)

In [ ]:
# Verificación de Fuga de Datos (Data Leakage)
def check_leakage(train: pd.DataFrame, val: pd.DataFrame, test: pd.DataFrame) -> None:
    sets = {
        'Train': set(train['text'].str.strip()),
        'Val':   set(val['text'].str.strip()),
        'Test':  set(test['text'].str.strip()),
    }
    pairs = [('Train', 'Val'), ('Train', 'Test'), ('Val', 'Test')]
    
    print('Intersección de Conjuntos')
    leakage_found = False
    for a, b in pairs:
        overlap = sets[a] & sets[b]
        status = 'OK (0 solapamientos)' if len(overlap) == 0 else f'LEAKAGE: {len(overlap)} textos duplicados'
        print(f'  {a:5s} ∩ {b:5s} : {status}')
        if overlap:
            leakage_found = True
            
    if not leakage_found:
        print('\nESTADO: Particiones estancas. Listo para modelado.')

check_leakage(ns_train, ns_val, ns_test)

In [ ]:
# Verificación de Estratificación (Proporción de Clases)
def split_distribution(splits: dict) -> pd.DataFrame:
    rows = {}
    for split_name, df in splits.items():
        counts = df['emotion'].value_counts().reindex(EMOTION_ORDER, fill_value=0)
        rows[split_name] = counts
    return pd.DataFrame(rows)

final_dist = split_distribution({'Train': ns_train, 'Val': ns_val, 'Test': ns_test})

print('Conteos Absolutos por Split')
display(final_dist.style.background_gradient(cmap='Blues', axis=None))

In [ ]:
# Heatmap de Mantenimiento Proporcional. Convertimos los conteos absolutos a porcentajes por columna (split)
pct_dist = final_dist.div(final_dist.sum(axis=0), axis=1) * 100

fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(
    pct_dist, 
    annot=True, 
    fmt='.1f', 
    cmap='YlOrRd',
    linewidths=0.5, 
    cbar_kws={'label': '% de representación en el split'},
    ax=ax
)

ax.set_title('Mantenimiento Proporcional de Clases (Estratificación)', fontweight='bold', pad=12)
ax.set_ylabel('Emoción', labelpad=8)
ax.set_xlabel('Subconjunto (Split)', labelpad=8)
plt.tight_layout()

save_custom_plot('final_split_stratification_heatmap')
plt.show()


## 7. Decisión Arquitectónica: Estrategia de Fine-Tuning

Se utilizará exclusivamente el conjunto de datos orgánico (data/processed/no_synthetic/) para la fase de entrenamiento y ajuste fino (fine-tuning) del modelo de clasificación, mitigando el desequilibrio residual mediante ponderación algorítmica (class weights). Esto es debido a los siguientes aspectos:

- **Priorización de la Calidad sobre la Cantidad (Mitigación de Ruido):** El análisis cualitativo reveló que las muestras sintéticas introducen artefactos estructurales severos (fórmulas repetitivas, prefijos anómalos y varianza dimensional nula). La inyección de estos datos degradaría la capacidad de generalización del modelo, provocando un sobreajuste (overfitting) hacia la sintaxis del modelo generador (Mistral) en lugar de capturar la naturaleza asimétrica del ciberacoso real.

- **Abordaje Algorítmico del Desequilibrio:** En lugar de forzar un balanceo a nivel de datos (Data-level), se aplicará una penalización a nivel de algoritmo. Configurar el parámetro class_weight='balanced' en el Trainer ajustará dinámicamente la función de pérdida (Loss), forzando a la red neuronal a penalizar con mayor severidad los errores cometidos en las clases minoritarias (fear, disgust, surprise).

- **Preservación de la Integridad del Corpus:** Al descartar la vía sintética, se garantiza que los subconjuntos de Entrenamiento, Validación y Prueba estén compuestos en un 100% por expresiones semánticas validadas (originales y traducciones humanas cruzadas). Esto asegura que las métricas de evaluación reflejen el rendimiento del modelo en un escenario de producción real.

- **Trazabilidad y Reproducibilidad Científica:** Las particiones que incluyen datos autogenerados (data/processed/with_synthetic/) se conservan en la arquitectura del proyecto como evidencia metodológica empírica de la experimentación fallida con Modelos de Lenguaje Pequeños (SLMs) para tareas de Synthetic Data Generation en dialectos informales.

Si tras la ejecución del fine-tuning las métricas de recuperación y precisión para las clases críticas resultan insuficientes (ej. $\text{F1-Score}_{\text{fear}} < 0.40$), se activarán las siguientes iteraciones:

- **Refinamiento del Prompt Engineering:** Diseñar un marco de generación más restrictivo, aplicando técnicas de Few-Shot Prompting con ejemplos de validación manual, forzando la salida en formato JSON estricto para evitar prefijos conversacionales.

- **Sustitución del Motor Generativo:** Escalar temporalmente la generación de datos desde un modelo local a APIs de modelos LLM que presentan mayor alineamiento en la simulación de dialectos adolescentes.